# B — Data Analysis Report

This notebook answers the analysis questions using the cleaned master data. Interpret results as associations, not proof of causation.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
tr=pd.read_csv(ROOT/'data/processed/master_train.csv',parse_dates=['pickup_hour']).dropna(subset=['trips'])
tr['hour']=tr.pickup_hour.dt.hour; tr['dow']=tr.pickup_hour.dt.dayofweek; tr['date']=tr.pickup_hour.dt.floor('D')
print('Rows:',len(tr),'period:',tr.pickup_hour.min(),'to',tr.pickup_hour.max(),'zones:',tr.zone.nunique())

## B1.1 Demand by zone

In [ ]:
zone=tr.groupby('zone').trips.agg(total_trips='sum',mean_trips='mean',zone_hours='count').sort_values('total_trips',ascending=False)
zone['city_share_pct']=100*zone.total_trips/zone.total_trips.sum()
display(zone.round(2))

## B1.2 Hour-of-day profiles and zone types

In [ ]:
profile=tr.groupby(['zone','hour']).trips.mean().unstack()
display(profile.round(1))
print('Peak hour by zone:',profile.idxmax(axis=1).to_dict())

## B1.3 Weekday vs weekend

In [ ]:
tr['day_type']=np.where(tr.dow>=5,'weekend','weekday')
w=tr.groupby(['zone','day_type']).trips.mean().unstack(); w['weekend_to_weekday_ratio']=w['weekend']/w['weekday']; display(w.round(3))

## B1.4 Trend

In [ ]:
weekly=tr.set_index('pickup_hour').trips.resample('W').sum(); display(weekly.to_frame('weekly_trips').tail(20)); print('First weekly total:',weekly.iloc[0],'last weekly total:',weekly.iloc[-1])

## B2.1 Weather clock check

In [ ]:
clock=pd.read_csv(ROOT/'reports/B2_1_clock_shift_test.csv'); display(clock)
print('Reminder: weather ISO timestamps with Z mean UTC and are converted to EAT; inspect correlation sensitivity as a diagnostic, not the sole basis for clock selection.')

## B2.2–B2.3 Rain effect

In [ ]:
rain=tr.groupby('rain_class',observed=False).trips.agg(['mean','count']); display(rain)
print('Rain comparisons are observational and can be confounded by hour, weekday, and zone.')

## B3.1–B3.4 Events and holidays

In [ ]:
events=pd.read_csv(ROOT/'reports/B3_cleaned_events.csv'); display(events.groupby(['event_type','status_clean']).size().rename('events').to_frame())
print('Cancelled events are excluded from predictive event-window features. Sparse event matching should be reported as a limitation.')

## B4.1–B4.3 Operations and data quality

In [ ]:
print(pd.read_csv(ROOT/'reports/B4_operational_correlations.csv').round(3))
print('Gap summary:'); display(pd.read_csv(ROOT/'reports/B4_gaps_by_zone.csv').head(12))
print('Operational columns are for historical analysis only; they are unavailable for future test hours and are excluded from model features.')